# 📅 2025-04-28 (화요일) 개발 노트 : 데이터 정합성 충돌 해결 및 '토큰 다이어트' 기반 지표 확장 파이프라인 구축

## 🎯 오늘의 목표
- [x] 기획안(52개)과 실제 추출 데이터(42개) 간의 스키마 불일치 원인 파악
- [x] 기존 데이터 유실 없이 18개 추가 지표를 추출할 최적의 아키텍처 수립
- [x] API 비용 절감을 위한 '토큰 다이어트' Batch 추출 및 병합(Merge) 프롬프트 작성

## 🛠 진행 상황 및 핵심 기록

### 1. 데이터 스키마 팩트 체크 (Single Source of Truth)
- 실제 추출된 `merged_games.jsonl` 데이터 구조 전수 조사 진행.
- **결과:** 수치 31개 + 태그 9개 + 평가 2개 = **총 42개 지표**로 구성됨을 확인. (기존 기획의 `Immersion`, `System` 카테고리 누락 확인)

### 2. 60개 마스터 데이터 확장을 위한 추가 지표(18개) 라인업 확정
- **복구 지표 (14개):** `build_variety`, `tutorial_quality`, `ui_ux_polish`, `puzzle_complexity` 등 기존 기획 누락분 복구.
- **신규 보완 지표 (4개):** 추천 엔진의 완성도를 높이기 위해 핵심 지표 추가.
  - `narrative_depth` (서사 깊이)
  - `replay_value` (다회차 가치)
  - `endgame_content` (엔드게임 볼륨)
  - `monetization_fairness` (과금 모델 합리성)

### 3. '토큰 다이어트(Token Diet)' 전략 설계
- **배경:** 4,190개 게임의 원본 스팀 리뷰를 다시 GPT API에 태우면 막대한 비용 발생.
- **전략:** 이미 추출된 42개짜리 JSONL 파일에서 짧은 요약본(`analysis_summary`, `core_loop` 등)만 텍스트로 추출하여 이를 기반으로 18개 지표만 추론하도록 시스템 설계. (Input 토큰 비용 95% 이상 절감)

## 🚨 트러블슈팅 (문제 및 해결)

### 1. 지표 차원 불일치로 인한 DB 적재(Load) 실패
- **문제:** 기획안(52개) 기준으로 작성된 `load_games.py`가 실제 데이터(42개)를 파싱하지 못하고 `KeyError` 및 필드 누락 에러 발생.
- **원인:** GPT-5.4 Batch 추출 시, 최신 기획안(52개)이 아닌 과거 버전(42개)의 JSON 스키마 프롬프트를 사용하여 데이터가 생성됨. 기획과 실제 데이터 간의 '버전 불일치'.
- **해결:** 기존 42개 데이터를 폐기하거나 코드를 하향 패치하는 타협안 대신, 기존 데이터를 보존하면서 **부족한 18개 지표만 저비용으로 추가 추출하여 병합(Merge)하는 파이프라인 스크립트 작성**으로 해결 방향 전환.

## 💡 인사이트 및 다음 할 일

### 📝 배운 점
- **"데이터가 왕이다"**: 데이터 파이프라인 구축 시, 기획서보다 **'실제로 추출된 데이터의 형태'**가 Single Source of Truth가 되어야 함.
- **프롬프트 버전 관리**: 대규모 Batch API를 태울 때는 프롬프트(JSON Schema)의 버전 관리가 인프라 아키텍처만큼 중요함.
- **엔지니어링적 사고**: 비용(API 과금)의 한계에 부딪혔을 때, 이를 우회하는 '요약본 재활용(토큰 다이어트)' 전략을 통해 가성비와 품질을 모두 잡을 수 있었음.

### 🚀 다음 할 일
- [ ] 1. 클로드가 작성한 `make_diet_batch.py`를 실행하여 OpenAI Batch API 요청 전송.
- [ ] 2. Batch 결과 수신 후 `merge_metrics.py`를 통해 60개 마스터 JSONL 데이터로 병합.
- [ ] 3. `validate_merge.py`를 실행하여 4,190개 데이터의 '49개 수치 지표' 정합성 100% 달성 여부 검증.
- [ ] 4. Django DB 마이그레이션 (`pure_embedding` 49차원으로 갱신) 및 최종 데이터 적재.